In [1]:
import os
import io
import zipfile
from pathlib import Path

import boto3

In [ ]:
AWS_ACCESS_KEY_ID = "ACCESS KEY"
AWS_SECRET_ACCESS_KEY = "SECRET KEY"
BUCKET_NAME = os.getenv("BUCKET_NAME", "giggso-florida-loan-data-share")

if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
    raise RuntimeError(
        "Missing AWS credentials. Please set AWS_ACCESS_KEY_ID and "
        "AWS_SECRET_ACCESS_KEY in your environment or .env file."
    )

if not BUCKET_NAME:
    raise RuntimeError(
        "Missing bucket name. Please set BUCKET_NAME in your environment or .env file."
    )

# --- Prepare local output directory ---
output_base_dir = Path.cwd().parent / "data" / "applicant_documents"
output_base_dir.mkdir(parents=True, exist_ok=True)
print(f"Files will be saved under: {output_base_dir}")

Files will be saved under: d:\FIU\Capstone II\capstone\data\applicant_documents


In [3]:

# --- Create S3 client ---
s3_client = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)

# --- List ZIP files in the bucket ---
paginator = s3_client.get_paginator("list_objects_v2")
zip_keys = []

print(f"Listing ZIP files in bucket '{BUCKET_NAME}'...")
for page in paginator.paginate(Bucket=BUCKET_NAME):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if key.lower().endswith(".zip"):
            zip_keys.append(key)

zip_keys.sort()
print(f"Found {len(zip_keys)} ZIP files.")


Listing ZIP files in bucket 'giggso-florida-loan-data-share'...
Found 4 ZIP files.


In [4]:

# --- Download and extract each ZIP ---
saved_paths = []

for idx, zip_key in enumerate(zip_keys, start=1):
    print(f"\n⬇ Downloading & extracting applicant {idx}: {zip_key}")
    zip_obj = s3_client.get_object(Bucket=BUCKET_NAME, Key=zip_key)
    zip_buffer = io.BytesIO(zip_obj["Body"].read())

    applicant_dir = output_base_dir / f"applicant_{idx}"
    applicant_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_buffer, "r") as z:
        for member in z.infolist():
            # Skip directories inside the zip
            if member.is_dir():
                continue

            # Only keep the filename part (ignore internal folders)
            filename = Path(member.filename).name
            local_path = applicant_dir / filename

            with z.open(member) as source, open(local_path, "wb") as target:
                target.write(source.read())

            saved_paths.append(local_path)
            print(f"  Saved: {local_path}")

print(f"\n✅ Done! Saved {len(saved_paths)} documents under {output_base_dir}")


⬇ Downloading & extracting applicant 1: Sample data files/GUL20240146-20241011T000608Z-001 (1).zip
  Saved: d:\FIU\Capstone II\capstone\data\applicant_documents\applicant_1\DISBURSEMENT MEMO_48.pdf
  Saved: d:\FIU\Capstone II\capstone\data\applicant_documents\applicant_1\OCR DOCUMENTS_28.pdf
  Saved: d:\FIU\Capstone II\capstone\data\applicant_documents\applicant_1\INSURANCE FORMS_44.pdf
  Saved: d:\FIU\Capstone II\capstone\data\applicant_documents\applicant_1\TECHNICAL REPORTS_65.pdf
  Saved: d:\FIU\Capstone II\capstone\data\applicant_documents\applicant_1\DRAFT SALE DEED - VETTED_63.pdf
  Saved: d:\FIU\Capstone II\capstone\data\applicant_documents\applicant_1\LEGAL APPRAISAL REPORT_66.pdf
  Saved: d:\FIU\Capstone II\capstone\data\applicant_documents\applicant_1\MOTD CHALLAN AND DRAFT COPY - VETTED_38.pdf
  Saved: d:\FIU\Capstone II\capstone\data\applicant_documents\applicant_1\MAIL CONFIRMATION FROM LEGAL OFFICERS_32.pdf
  Saved: d:\FIU\Capstone II\capstone\data\applicant_documents\a

In [ ]:
import math
from collections import Counter
from google.cloud import vision
import re

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = "path to json file with credentials"
WORD = re.compile(r"\w+")

def detect_text_from_local_pdf(pdf_path: str, timeout: int = 300):
    """
    OCR a local PDF using Google Cloud Vision API (async processing).

    Args:
        pdf_path: Local path to the PDF file.
        timeout: How long to wait for async Vision operation.

    Returns:
        A list of strings (one per page or annotation chunk).
    """

    client = vision.ImageAnnotatorClient()

    # Read the PDF bytes
    with open(pdf_path, "rb") as f:
        pdf_bytes = f.read()

    mime_type = "application/pdf"
    feature = vision.Feature(type_=vision.Feature.Type.DOCUMENT_TEXT_DETECTION)

    input_config = vision.InputConfig(
        content=pdf_bytes,
        mime_type=mime_type
    )

    request = vision.AsyncAnnotateFileRequest(
        features=[feature],
        input_config=input_config
    )

    print(f"🔍 Running OCR on local PDF: {pdf_path}")
    operation = client.async_batch_annotate_files(requests=[request])

    # Wait for async operation to complete
    response = operation.result(timeout=timeout)

    ocr_text = []

    # response.responses → list of FileAnnotationResponse
    for file_response in response.responses:
        # file_response.responses → per-page annotation results
        for page_response in file_response.responses:
            if page_response.error.message:
                raise Exception(
                    f"Vision API error: {page_response.error.message}\n"
                    "See https://cloud.google.com/apis/design/errors"
                )

            if page_response.full_text_annotation:
                ocr_text.append(page_response.full_text_annotation.text)

    print(f"✅ OCR completed for: {pdf_path}")
    return ocr_text

In [10]:
ocr_pages = detect_text_from_local_pdf(str(saved_paths[0]))

🔍 Running OCR on local PDF: d:\FIU\Capstone II\capstone\data\applicant_documents\applicant_1\DISBURSEMENT MEMO_48.pdf


InvalidArgument: 400 GcsSource is required.

In [ ]:
print(ocr_pages)

WindowsPath('d:/FIU/Capstone II/capstone/data/applicant_documents/applicant_1/DISBURSEMENT MEMO_48.pdf')